In [1]:
!pip install torch_scatter torcheeg torch_geometric -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.4/251.4 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.5/231.5 kB 13.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.1/295.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.2/115.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 59.0 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.utils as utils
from torch.utils.data import DataLoader, Subset,WeightedRandomSampler
from torcheeg.models import CCNN
from torcheeg import transforms
from torcheeg.transforms import ToGrid
from torcheeg.datasets import SEEDIVDataset,SEEDIVFeatureDataset
from torcheeg.datasets.constants import SEED_IV_CHANNEL_LOCATION_DICT
from torcheeg.transforms import ToG
from torcheeg.datasets.constants import SEED_IV_ADJACENCY_MATRIX
from torcheeg.models import DGCNN
from torch.optim.lr_scheduler import CosineAnnealingLR
# --- THE MAIN SUBJECT LOOP ---

import torch_geometric.loader as geom_loader # Special loader for graphs
import copy
import scipy.signal as signal
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
import numpy as np
import os

In [3]:
# 1. Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [7]:
# This DataFrame contains columns like: subject_id, trial_id, emotion, etc.
meta_info = dataset.info 
all_subjects = sorted(meta_info['subject_id'].unique())
print(f"Subjects Found: {all_subjects}")

Subjects Found: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


In [22]:
dataset = SEEDIVFeatureDataset(
    io_path='./tmp_out/seed_iv_features_grid2',
    root_path='/kaggle/input/seed-iv/eeg_feature_smooth',
    feature=['de_LDS'],
    offline_transform=transforms.Compose([
        transforms.To2d(),
        transforms.ToTensor()
    ]),
    label_transform=transforms.Select('emotion')
)

[2026-02-28 19:42:43] INFO (torcheeg/MainThread) 🔍 | Processing EEG data. Processed EEG data has been cached to ./tmp_out/seed_iv_features_grid2.
[2026-02-28 19:42:43] INFO (torcheeg/MainThread) ⏳ | Monitoring the detailed processing of a record for debugging. The processing of other records will only be reported in percentage to keep it clean.
[PROCESS]:   0%|          | 0/45 [00:00<?, ?it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 0it [00:00, ?it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 8it [00:00, 78.31it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 33it [00:00, 175.27it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 57it [00:00, 201.82it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 80it [00:00, 209.61it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4_20151111.mat]: 104it [00:00, 217.64it/s]
[RECORD /kaggle/input/seed-iv/eeg_feature_smooth/1/4

In [23]:
dataset

SEEDIVFeatureDataset(
    io_path='./tmp_out/seed_iv_features_grid2',
    io_size=1048576,
    io_mode='lmdb',
    root_path='/kaggle/input/seed-iv/eeg_feature_smooth',
    feature=['de_LDS'],
    num_channel=62,
    online_transform=None,
    offline_transform=Compose(
    To2d(apply_to_baseline=False),
    ToTensor(apply_to_baseline=False)
),
    label_transform=Select(key='emotion'),
    before_trial=None,
    after_trial=None,
    num_worker=0,
    verbose=True
)
length=37575

In [30]:
# Parameters
EPOCHS = 100
BATCH_SIZE = 64
LR = 1e-5
PATIENCE_LR = 3
REDUCE_FACTOR = 0.5
PATIENCE_ES = 25
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1

In [31]:
import os
from sklearn.model_selection import train_test_split

In [32]:
# Subject-Dependent Loop
import os
from sklearn.model_selection import train_test_split

# Create a directory to save the best models
if not os.path.exists('saved_models'):
    os.makedirs('saved_models')

subject_results = {}

print("Start Subject-Dependent Evaluation...")

meta_info = dataset.info 
all_subjects = sorted(meta_info['subject_id'].unique())
original_labels = meta_info['emotion'].unique()
for subject_id in all_subjects:
    print(f"\n{'='*30} Processing Subject {subject_id} {'='*30}")
    
    # Create a unique ID for each recording (session + trial)
    # Since SEED IV repeats trial IDs (1-24) across sessions, 
    # merging them ensures we recognize all 72 trials per subject
    subject_df = meta_info[meta_info['subject_id'] == subject_id].copy()

    # # -------------------------------------------------------
    # # خطوة الـ Standardization (Subject-wise)
    # # -------------------------------------------------------
    # # 1. استخراج الـ indices الخاصة بهذا الـ Subject فقط من الـ dataset الأصلي
    # subject_indices = subject_df.index.tolist()
    
    # # 2. الحصول على البيانات الخام (Raw Data) لهذا الـ Subject
    # # نفترض أن dataset.data هي مصفوفة numpy بـ shape (N, 62, 5)
    # # سنقوم بعمل تسطيح (Flatten) للبيانات لتناسب الـ Scaler
    # raw_data = np.array([dataset.feature[i] for i in subject_indices]) # shape: (Samples, 62, 5)
    # # raw_data = dataset.feature[subject_indices] # shape: (Samples, 62, 5)
    # n_samples, n_channels, n_bands = raw_data.shape
    # raw_data_reshaped = raw_data.reshape(n_samples, -1) # (Samples, 310)

    # # 3. حساب المتوسط والانحراف المعياري وتطبيق التحويل
    # scaler = StandardScaler()
    # scaled_data = scaler.fit_transform(raw_data_reshaped)
    
    # # 4. إعادة البيانات لشكلها الأصلي وتحديث الـ dataset (مؤقتاً لهذا الـ Subject)
    # dataset.data[subject_indices] = scaled_data.reshape(n_samples, n_channels, n_bands).astype(np.float32)
    # # -------------------------------------------------------
    
    subject_df['unique_run_id'] = subject_df['session_id'].astype(str) + "_" + subject_df['trial_id'].astype(str)
    
    # Extract labels per unique run to ensure balanced split
    # We get a single label for each unique run (session_trial)
    run_info = subject_df[['unique_run_id', 'emotion']].drop_duplicates()
    all_runs = run_info['unique_run_id'].values
    all_labels = run_info['emotion'].values
    
    # Stratified Split at the RUN level 
    # This keeps 80/20 ratio and ensures each class is represented in the test set
    train_runs, test_runs = train_test_split(
        all_runs, 
        test_size=0.20, 
        random_state=42, 
        stratify=all_labels
    )
    
    print(f"Total Unique Videos (Across 3 Sessions): {len(all_runs)}")
    print(f"Training on: {len(train_runs)} | Testing on: {len(test_runs)}")
    
    # Extract indices (Zero Leakage Guaranteed)
    train_indices = subject_df[subject_df['unique_run_id'].isin(train_runs)].index.tolist()
    test_indices = subject_df[subject_df['unique_run_id'].isin(test_runs)].index.tolist()
    
    # Dataset Subsets and Loaders
    train_set = Subset(dataset, train_indices)
    test_set = Subset(dataset, test_indices)
    
    # Weighted Sampler for further balancing during training 
    y_train = meta_info.iloc[train_indices]['emotion'].values
    class_counts = np.bincount(y_train)
    class_weights = 1. / (class_counts + 1e-6)
    sample_weights = [class_weights[y] for y in y_train]
    
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=False, sampler=sampler)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)
    
    # --- 3. Initialize Model ---
    model = BFENet_Full(num_classes=4).to(device)
    
    criterion = nn.CrossEntropyLoss( label_smoothing = LABEL_SMOOTHING)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    # scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=REDUCE_FACTOR, patience=PATIENCE_LR)

    # optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    debug_done = False

    # Training Loop 
    best_val_acc = 0.0
    counter = 0 # For early stopping
    for epoch in range(EPOCHS):
        model.train()
        correct = 0
        total = 0
        train_loss = 0
        
        for X, y in train_loader:
            X = X.to(device)
            y = y.to(device).long()

            # if not debug_done:
            #     print("DEBUG X shape:", X.shape)
            #     print("DEBUG y shape:", y.shape)
            #     debug_done = True

            
            if len(X.shape) == 4: X = X.squeeze(1) 

            # 2. الـ Standardization (هنا السحر!)
            # بنحسب المتوسط والانحراف لكل عينة في الـ Batch
            # dim=(1,2) عشان نحسب لكل قناة وتردد بالنسبة لكل Sample
            mean = X.mean(dim=(1, 2), keepdim=True)
            std = X.std(dim=(1, 2), keepdim=True) + 1e-6
            X = (X - mean) / std # تحويل البيانات لـ Standard Normal Distribution

            # print("DEBUG X shape:", X.shape)
            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()
            
       
        avg_train_loss = train_loss / len(train_loader)
        train_acc = 100 * correct / total
            
        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0
        with torch.no_grad():
            for X, y in test_loader:
                X = X.to(device)
                y = y.to(device).long()
                if len(X.shape) == 4: X = X.squeeze(1)

                
                mean = X.mean(dim=(1, 2), keepdim=True)
                std = X.std(dim=(1, 2), keepdim=True) + 1e-6
                X = (X - mean) / std # تحويل البيانات لـ Standard Normal Distribution
                
                outputs = model(X)
                val_loss += criterion(outputs, y).item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += y.size(0)
                val_correct += (predicted == y).sum().item()
        
        val_acc = 100 * val_correct / val_total
        avg_val_loss = val_loss / len(test_loader)
        #scheduler.step(avg_val_loss)
        scheduler.step()

        print(f"Epoch {epoch+1:02d} | "
              f"Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.2f}%")
        
        # Early Stopping Check
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            counter = 0

            save_path = f"saved_models/subject_{subject_id}_best.pth"
            torch.save(model.state_dict(), save_path)
            
        else:
            counter += 1
            if counter >= PATIENCE_ES:
                print(f"Early stopping at epoch {epoch}")
                break
                
    print(f"Subject {subject_id} Best Acc: {best_val_acc:.2f}%")
    subject_results[subject_id] = best_val_acc

# ================= FINAL REPORT =================
print("\n" + "="*40)
print("FINAL RESULTS REPORT ")
print("="*40)

all_accuracies = list(subject_results.values())

for sub_id in sorted(subject_results.keys()):
    print(f"Subject {sub_id}: {subject_results[sub_id]:.2f}%")

print(f"\nAverage Accuracy: {np.mean(all_accuracies):.2f}%")

if len(subject_results) > 0:
    best_sub_id = max(subject_results, key=subject_results.get)
    print(f"HIGHEST ACCURACY: Subject {best_sub_id} ({subject_results[best_sub_id]:.2f}%)")
print("="*40)

Start Subject-Dependent Evaluation...

============================== Processing Subject 1 ==============================
Total Unique Videos (Across 3 Sessions): 72
Training on: 57 | Testing on: 15
Epoch 01 | Train Loss: 1.3884 Acc: 24.70% | Val Loss: 1.3955 Acc: 19.42%
Epoch 02 | Train Loss: 1.3839 Acc: 26.21% | Val Loss: 1.3885 Acc: 19.42%
Epoch 03 | Train Loss: 1.3790 Acc: 27.87% | Val Loss: 1.3815 Acc: 27.84%
Epoch 04 | Train Loss: 1.3637 Acc: 42.43% | Val Loss: 1.3565 Acc: 44.67%
Epoch 05 | Train Loss: 1.3377 Acc: 39.37% | Val Loss: 1.3594 Acc: 29.21%
Epoch 06 | Train Loss: 1.2701 Acc: 44.77% | Val Loss: 1.3847 Acc: 24.40%
Epoch 07 | Train Loss: 1.1515 Acc: 51.43% | Val Loss: 1.2472 Acc: 38.14%
Epoch 08 | Train Loss: 1.0475 Acc: 58.66% | Val Loss: 1.2733 Acc: 39.86%
Epoch 09 | Train Loss: 0.9569 Acc: 68.17% | Val Loss: 1.1313 Acc: 46.91%
Epoch 10 | Train Loss: 0.8809 Acc: 76.76% | Val Loss: 1.0244 Acc: 52.75%
Epoch 11 | Train Loss: 0.7991 Acc: 81.70% | Val Loss: 1.0091 Acc: 52.06

In [33]:
# ================= FINAL REPORT =================
print("\n" + "="*40)
print("FINAL RESULTS REPORT ")
print("="*40)

all_accuracies = list(subject_results.values())

for sub_id in sorted(subject_results.keys()):
    print(f"Subject {sub_id}: {subject_results[sub_id]:.2f}%")

print(f"\nAverage Accuracy: {np.mean(all_accuracies):.2f}%")

if len(subject_results) > 0:
    best_sub_id = max(subject_results, key=subject_results.get)
    print(f"HIGHEST ACCURACY: Subject {best_sub_id} ({subject_results[best_sub_id]:.2f}%)")
print("="*40)


FINAL RESULTS REPORT 
Subject 1: 81.27%
Subject 2: 78.87%
Subject 3: 80.24%
Subject 4: 86.77%
Subject 5: 69.93%
Subject 6: 69.24%
Subject 7: 78.52%
Subject 8: 94.50%
Subject 9: 66.32%
Subject 10: 82.82%
Subject 11: 38.49%
Subject 12: 48.45%
Subject 13: 57.04%
Subject 14: 62.03%
Subject 15: 88.49%

Average Accuracy: 72.20%
HIGHEST ACCURACY: Subject 8 (94.50%)
